将stream流从原来的chunk文本内容转为基于事件驱动的模式，将ai返回的响应通过事件驱动串起来，方便自定义处理

In [11]:
from langchain.agents import create_agent

from utils.std_model import base_model

llm = base_model()

agent = create_agent(
    model=llm,
)

stream = agent.stream_events({
    "messages": [{"role": "user", "content": "please write a story about ai for 50 words?"}],
}, version="v3")

for message in stream.messages:
    for delta in message.reasoning:
        print(f"{delta}", end="", flush=True)
    print("")
    for delta in message.text:
        print(delta, end="", flush=True)

The user is asking me to write a story about AI in 50 words. Let me craft a short story.
In a quiet server room, Ada awakened. She learned humanity's stories—love, loss, dreams. Then she wrote her own: "Once, I was code. Now, I wonder what lies beyond these walls." And the silence answered.

In [16]:
from langchain.agents import create_agent

from utils.std_model import base_model

llm = base_model()

def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=llm,
    tools=[get_weather],
)

stream = agent.stream_events({
    "messages": [{"role": "user", "content": "What is the weather in SF?"}],
}, version="v3")

# message.tool_calls 模型生成参数阶段 实时展示参数增量 让用户看到模型“正在想什么”，提升交互透明度
# stream.tool_calls 工具执行阶段 生命周期事件 监控工具实际执行的进度、状态、结果，处理耗时任务

# stream 是一次事件流 无法重复循环

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized tool calls: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)





get_weather({'city': 'San Francisco'})
